# Inbox Signal: walkthrough
From a raw message to an "important or not" decision, step by step. The Streamlit app runs the same code.

In [1]:
import sys, warnings
sys.path.insert(0, "..")          # so we can import the project package from the notebook
warnings.filterwarnings("ignore")
import pandas as pd
pd.set_option("display.max_colwidth", 120)

## Step 1 · Load messages
Every source (IMAP, .eml, .mbox, iMessage, Android XML, CSV) is converted to the same `Message` shape.

In [2]:
from inbox_signal import load_any

msgs = load_any("../data/sample/messages.json")
pd.DataFrame([m.to_dict() for m in msgs])[["channel", "sender_name", "subject", "body"]].head(8)

,channel,sender_name,subject,body
0,sms,Mom,,"Hi sweetie, are you still coming to dinner Sunday? Grandma wants to know by tonight so she can shop."
1,email,Priya Raman,Q3 deck before client call,"Hi, can you send me the updated Q3 deck before the 2pm call with Brightline today? I need to review the pricing slid..."
2,email,City Utilities,Your bill is past due,Your electricity bill of $142.80 is past due. Please pay by Oct 14 to avoid a $25 late fee and service interruption.
3,sms,72166,,Your Chase verification code is 482913. Don't share it with anyone. We'll never call to ask for it.
4,email,ShopWave,FLASH SALE 40% off everything,This weekend only! 40% off sitewide plus free shipping. Use promo code FALL40. Shop now. Unsubscribe | Manage prefer...
5,email,Instagram,jake_m liked your photo,jake_m and 3 others liked your photo. See what's new in your feed.
6,email,ParcelNow,Your order has shipped,Your order #1182 has shipped and will arrive Thursday. Track your package anytime.
7,sms,Dr. Alvarez Dental,,Reminder: your dentist appointment is tomorrow at 9:30am with Dr. Alvarez. Reply C to confirm or call to reschedule.


## Step 2 · Load a zero-shot model from Hugging Face
Zero-shot classification asks the model: *how likely is "This message is a bill, payment, or
banking notice." given this text?* You write the categories in plain English and need **no
training data**.

In [3]:
from transformers import pipeline

zs = pipeline("zero-shot-classification", model="MoritzLaurer/deberta-v3-base-zeroshot-v2.0")
out = zs("Your electricity bill of $142.80 is past due. Pay by Oct 14 to avoid a late fee.",
         candidate_labels=["a bill, payment, or banking notice", "a newsletter or marketing promotion",
                           "a personal message from a friend or family member"],
         hypothesis_template="This message is {}.")
dict(zip(out["labels"], [round(s, 3) for s in out["scores"]]))

Device set to use cpu


{'a bill, payment, or banking notice': 0.999,
 'a newsletter or marketing promotion': 0.0,
 'a personal message from a friend or family member': 0.0}

## Step 3 · Rule-based signals (no ML, fully explainable)

In [4]:
from inbox_signal import features

features.extract("URGENT: can you send the deck before the 2pm call today?",
                 sender="priya@company.com", sender_name="Priya", vips=["priya@company.com"])

{'urgent': 'URGENT',
 'when': '2pm',
 'money': '',
 'question': 'can you send the deck before the 2pm call today?',
 'bulk': '',
 'noreply': False,
 'shortcode': False,
 'otp': False,
 'shortener': '',
 'vip': 'priya@company.com'}

## Step 4 · Score everything: model + rules → 0-100 with reasons

In [5]:
from inbox_signal import score_messages

scored = score_messages(msgs, vips=["Mom"], use_personal=False)
scored[["from", "category", "score", "tier", "gist"]].head(12)

Device set to use cpu


,from,category,score,tier,gist
0,Mom,personal,100.0,important,are you still coming to dinner Sunday?
1,Priya Raman,work,100.0,important,Q3 deck before client call - can you send me the updated Q3 deck before the 2pm call with Brightline today?
2,Prof. Helen Wu,work,100.0,important,Recommendation letter - I'm happy to write your recommendation letter.
3,Alicia Gomez,work,100.0,important,Interview availability - Data Scientist role - thanks for applying to the Data Scientist position at Lumen Analytics.
4,City Utilities,bill,97.8,important,Your bill is past due - Your electricity bill of $142.80 is past due.
5,Mom,personal,93.7,important,Also dad says happy early birthday 🎂
6,Aunt Ruth,personal,89.2,important,"Call me when you can, it's about Grandpa."
7,Sam (roommate),work,85.1,important,"yo the landlord is coming at 4 today to fix the sink, can you be home?"
8,Dr. Alvarez Dental,appointment,82.0,important,Reminder: your dentist appointment is tomorrow at 9:30am with Dr.
9,Jordan Lee,appointment,74.0,important,Wedding - can you be a groomsman? - Taylor and I set the date for June 14.


In [6]:
row = scored.iloc[0]
print(row["from"], "→", row["score"])
for r in row["reasons"]:
    print("  -", r)

Mom → 100.0
  - Looks like: personal (86%)
  - From a VIP: Mom
  - Urgent wording: “by tonight”
  - Mentions a time: “Sunday”
  - Asks you a direct question


## Step 5 · How accurate is it? Compare with hand labels

In [7]:
import json
from sklearn.metrics import classification_report

expected = {m["id"]: m["expected_important"] for m in json.load(open("../data/sample/messages.json"))}
y_true = scored["id"].map(expected)
y_pred = (scored["tier"] == "important").astype(int)
print(classification_report(y_true, y_pred, target_names=["not important", "important"]))

               precision    recall  f1-score   support

not important       0.93      0.81      0.87        16
    important       0.81      0.93      0.87        14

     accuracy                           0.87        30
    macro avg       0.87      0.87      0.87        30
 weighted avg       0.87      0.87      0.87        30



## Step 6 · The digest you'd receive

In [8]:
from inbox_signal.digest import build_markdown
from IPython.display import Markdown

Markdown(build_markdown(scored))

# Inbox Signal · Thu Oct 08, 09:30 AM
**16 important** out of 30 messages · 13 filtered out

## Needs your attention
- 💬 **Mom** (personal, 100) - are you still coming to dinner Sunday?
  - _why: From a VIP: Mom; Urgent wording: “by tonight”; Mentions a time: “Sunday”_
- ✉️ **Priya Raman** (work, 100) - Q3 deck before client call - can you send me the updated Q3 deck before the 2pm call with Brightline today?
  - _why: Mentions a time: “2pm”; Asks you a direct question_
- ✉️ **Prof. Helen Wu** (work, 100) - Recommendation letter - I'm happy to write your recommendation letter.
  - _why: Mentions a time: “Monday”; Asks you a direct question_
- ✉️ **Alicia Gomez** (work, 100) - Interview availability - Data Scientist role - thanks for applying to the Data Scientist position at Lumen Analytics.
  - _why: Mentions a time: “Wednesday”; Asks you a direct question_
- ✉️ **City Utilities** (bill, 98) - Your bill is past due - Your electricity bill of $142.80 is past due.
  - _why: Urgent wording: “past due”; Mentions a time: “Oct 14”; Money involved: $142.80_
- 💬 **Mom** (personal, 94) - Also dad says happy early birthday 🎂
  - _why: From a VIP: Mom_
- 💬 **Aunt Ruth** (personal, 89) - Call me when you can, it's about Grandpa.
  - _why: Urgent wording: “Call me”; Mentions a time: “today”_
- 💬 **Sam (roommate)** (work, 85) - yo the landlord is coming at 4 today to fix the sink, can you be home?
  - _why: Mentions a time: “today”; Asks you a direct question_
- 💬 **Dr. Alvarez Dental** (appointment, 82) - Reminder: your dentist appointment is tomorrow at 9:30am with Dr.
  - _why: Mentions a time: “tomorrow”_
- ✉️ **Jordan Lee** (appointment, 74) - Wedding - can you be a groomsman? - Taylor and I set the date for June 14.
  - _why: Mentions a time: “June 14”; Asks you a direct question_
- 💬 **Sam (roommate)** (bill, 72) - rent is due on the 1st, venmo me $850 when you get a chance
  - _why: Money involved: $850_
- ✉️ **Northwind HR** (work, 72) - ACTION REQUIRED: Benefits enrollment closes Friday - Open enrollment for 2027 benefits closes this Friday, Oct 10.
  - _why: Urgent wording: “ACTION REQUIRED”; Mentions a time: “Friday”_
- ✉️ **First Bank** (security, 69) - Unusual sign-in attempt - We noticed a sign-in to your account from a new device in Denver, CO.
  - _why: Urgent wording: “immediately”_
- ✉️ **City Library** (bill, 66) - Items due soon - Reminder: 2 items are due on Oct 15.
  - _why: Mentions a time: “Oct 15”; Sent from a no-reply / notifications address_
- ✉️ **Google Calendar** (appointment, 65) - Updated invitation: Team sync moved to 3pm - The Team sync on Thursday has been moved from 10am to 3pm by Dana Whitfield.
  - _why: Mentions a time: “3pm”; Sent from a no-reply / notifications address_
- ✉️ **Maria Chen** (work, 56) - Re: budget numbers - Thanks - got it.
  - _why: Looks like: work (36%)_

## Worth a glance
- ✉️ Corner Cafe - Your receipt - Thanks for your order!

## Next
- `python scripts/make_dataset.py && python scripts/train.py --distilbert`: train and compare TF-IDF vs. fine-tuned DistilBERT
- `streamlit run app.py`: label messages 👍/👎 to train your personal model